# 🛒 Predicción Inteligente de Gasto en Clientes E-commerce
## Módulo 6: Aprendizaje de Máquina Supervisado — Alkemy
> **Objetivo:** Diseñar e implementar un modelo predictivo de regresión para estimar el monto de compra esperado por cliente.

---


## 📦 Importaciones y Configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#3B1F2B','#44BBA4']
print('✅ Librerías importadas correctamente')

## 📌 Lección 1: Fundamentos del Aprendizaje de Máquina

In [ ]:
# Definición del problema
print('Tipo de problema: REGRESIÓN SUPERVISADA')
print('Variable objetivo: monto_compra_promedio (valor continuo numérico)')
print()
print('Pipeline ML:')
etapas = ['1. Carga y EDA', '2. Preprocesamiento', '3. Split Train/Test',
          '4. Entrenamiento', '5. Evaluación (MAE/RMSE/R²)', '6. Optimización']
for e in etapas:
    print(f'  {e}')

## 📥 Carga de Datos

In [ ]:
df = pd.read_csv('dataset_ecommerce.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

## 📊 Análisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('EDA - Variables Numéricas', fontsize=14, fontweight='bold')

num_cols = ['monto_compra_promedio','edad','frecuencia_visitas_mes',
            'items_en_carrito','numero_compras_previas','tiempo_promedio_sesion_min']
for ax, col in zip(axes.flat, num_cols):
    ax.hist(df[col].dropna(), bins=30, color=COLORS[0], edgecolor='white', alpha=0.85)
    ax.set_title(col); ax.set_xlabel(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cat_cols = ['nivel_membresia','categoria_preferida','dispositivo']
for ax, col in zip(axes, cat_cols):
    df.groupby(col)['monto_compra_promedio'].mean().sort_values().plot(kind='barh', ax=ax, color=COLORS[1], alpha=0.85)
    ax.set_title(f'Monto promedio por {col}')
plt.tight_layout()
plt.show()

## 🔧 Lección 3: Preprocesamiento y Escalamiento

In [ ]:
# Imputación de valores nulos con mediana
for col in ['tiempo_promedio_sesion_min','calificacion_promedio_dada','dias_desde_ultima_compra']:
    df[col] = df[col].fillna(df[col].median())
print(f'Nulos restantes: {df.isnull().sum().sum()}')

# Eliminar outliers del target con IQR
Q1, Q3 = df['monto_compra_promedio'].quantile([0.25, 0.75])
IQR = Q3 - Q1
df = df[~((df['monto_compra_promedio'] < Q1-1.5*IQR) | (df['monto_compra_promedio'] > Q3+1.5*IQR))].reset_index(drop=True)
print(f'Filas tras eliminar outliers: {len(df)}')

# Codificación Label Encoding
cat_cols = ['genero','region','dispositivo','categoria_preferida','nivel_membresia']
df_enc = df.copy()
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

X = df_enc.drop('monto_compra_promedio', axis=1)
y = df_enc['monto_compra_promedio']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalamiento
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 📈 Lección 4: Regresiones

In [ ]:
# Regresión Lineal
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
print(f'Regresión Lineal → MAE:{mean_absolute_error(y_test,y_pred_lr):.2f}',
      f'RMSE:{np.sqrt(mean_squared_error(y_test,y_pred_lr)):.2f}',
      f'R²:{r2_score(y_test,y_pred_lr):.4f}')

# Regresión Polinomial grado 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_p = poly.fit_transform(X_train_sc)
X_ts_p = poly.transform(X_test_sc)
lr_poly = LinearRegression()
lr_poly.fit(X_tr_p, y_train)
y_pred_poly = lr_poly.predict(X_ts_p)
print(f'Polinomial(g=2) → MAE:{mean_absolute_error(y_test,y_pred_poly):.2f}',
      f'RMSE:{np.sqrt(mean_squared_error(y_test,y_pred_poly)):.2f}',
      f'R²:{r2_score(y_test,y_pred_poly):.4f}')

# Coeficientes
coef_df = pd.DataFrame({'Feature':X.columns,'Coef':np.abs(lr.coef_)}).sort_values('Coef',ascending=False)
coef_df.head(8)

## 🔵 Lección 5: KNN Regressor

In [ ]:
knn = KNeighborsRegressor(n_neighbors=7)
knn.fit(X_train_sc, y_train)
y_pred_knn = knn.predict(X_test_sc)
print(f'KNN(k=7) → MAE:{mean_absolute_error(y_test,y_pred_knn):.2f}',
      f'RMSE:{np.sqrt(mean_squared_error(y_test,y_pred_knn)):.2f}',
      f'R²:{r2_score(y_test,y_pred_knn):.4f}')

print()
print('Nota: KNN clasificador no aplica aquí porque el target es continuo.')
print('KNN Regressor promedia los k vecinos más cercanos para estimar el valor.')

## ✅ Lección 2: Validación Cruzada K-Folds

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_lr  = cross_val_score(LinearRegression(), X_train_sc, y_train, cv=kf, scoring='r2')
cv_knn = cross_val_score(KNeighborsRegressor(n_neighbors=7), X_train_sc, y_train, cv=kf, scoring='r2')

print(f'CV R² - Reg. Lineal: {cv_lr.mean():.4f} ± {cv_lr.std():.4f}')
print(f'CV R² - KNN:         {cv_knn.mean():.4f} ± {cv_knn.std():.4f}')

# Visualizar distribución de scores CV
fig, ax = plt.subplots(figsize=(8,4))
ax.boxplot([cv_lr, cv_knn], labels=['Reg. Lineal','KNN'], patch_artist=True,
           boxprops=dict(facecolor=COLORS[0], alpha=0.7))
ax.set_title('R² por Fold — Validación Cruzada K=5')
ax.set_ylabel('R²')
plt.tight_layout(); plt.show()

## ⚙️ Lección 7: Optimización — Ridge, Lasso y GridSearchCV

In [ ]:
# Ridge
ridge_gs = GridSearchCV(Ridge(), {'alpha':[0.01,0.1,1,10,100]}, cv=5, scoring='r2')
ridge_gs.fit(X_train_sc, y_train)
y_pred_ridge = ridge_gs.best_estimator_.predict(X_test_sc)
print(f'Ridge best alpha={ridge_gs.best_params_["alpha"]}',
      f'→ R²:{r2_score(y_test,y_pred_ridge):.4f}')

# Lasso
lasso_gs = GridSearchCV(Lasso(max_iter=5000), {'alpha':[0.01,0.1,1,10]}, cv=5, scoring='r2')
lasso_gs.fit(X_train_sc, y_train)
y_pred_lasso = lasso_gs.best_estimator_.predict(X_test_sc)
print(f'Lasso best alpha={lasso_gs.best_params_["alpha"]}',
      f'→ R²:{r2_score(y_test,y_pred_lasso):.4f}')

## 🚀 Lección 8: Gradient Boosting

In [ ]:
gb_gs = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {'n_estimators':[100,200],'learning_rate':[0.05,0.1],'max_depth':[3,4]},
    cv=3, scoring='r2', n_jobs=-1
)
gb_gs.fit(X_train_sc, y_train)
best_gb = gb_gs.best_estimator_
y_pred_gb = best_gb.predict(X_test_sc)

mae_gb  = mean_absolute_error(y_test, y_pred_gb)
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb   = r2_score(y_test, y_pred_gb)
print(f'GB {gb_gs.best_params_}')
print(f'MAE:{mae_gb:.2f} | RMSE:{rmse_gb:.2f} | R²:{r2_gb:.4f}')

## 📊 Lección 6: Tabla Comparativa de Métricas

In [ ]:
mae_lr_v  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr_v = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr_v   = r2_score(y_test, y_pred_lr)

results = pd.DataFrame({
    'Modelo':  ['Reg. Lineal','Reg. Polinomial','KNN (k=7)','Ridge','Lasso','Gradient Boosting'],
    'MAE':     [mae_lr_v,
                mean_absolute_error(y_test,y_pred_poly),
                mean_absolute_error(y_test,y_pred_knn),
                mean_absolute_error(y_test,y_pred_ridge),
                mean_absolute_error(y_test,y_pred_lasso),
                mae_gb],
    'RMSE':    [rmse_lr_v,
                np.sqrt(mean_squared_error(y_test,y_pred_poly)),
                np.sqrt(mean_squared_error(y_test,y_pred_knn)),
                np.sqrt(mean_squared_error(y_test,y_pred_ridge)),
                np.sqrt(mean_squared_error(y_test,y_pred_lasso)),
                rmse_gb],
    'R²':      [r2_lr_v,
                r2_score(y_test,y_pred_poly),
                r2_score(y_test,y_pred_knn),
                r2_score(y_test,y_pred_ridge),
                r2_score(y_test,y_pred_lasso),
                r2_gb]
}).round(4)
results.style.highlight_max(subset=['R²'], color='#d4edda').highlight_min(subset=['MAE','RMSE'], color='#d4edda')

In [ ]:
# Gráfico comparativo de R²
fig, axes = plt.subplots(1, 3, figsize=(17,5))
fig.suptitle('Comparación de Métricas por Modelo', fontsize=13, fontweight='bold')
for ax, metric in zip(axes, ['MAE','RMSE','R²']):
    bars = ax.bar(results['Modelo'], results[metric], color=COLORS[:6], alpha=0.85, edgecolor='white')
    ax.set_title(metric); ax.tick_params(axis='x', rotation=45)
    for bar, val in zip(bars, results[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.3f}', ha='center', fontsize=8)
    if metric == 'R²': ax.set_ylim(0,1.1)
plt.tight_layout(); plt.show()

In [ ]:
# Real vs Predicho
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for ax, (name, yp) in zip(axes, [('Reg. Lineal', y_pred_lr), ('Gradient Boosting', y_pred_gb)]):
    ax.scatter(y_test, yp, alpha=0.35, color=COLORS[0], s=20)
    lims=[min(y_test.min(),yp.min()), max(y_test.max(),yp.max())]
    ax.plot(lims,lims,'r--',linewidth=1.5,label='Perfecta')
    ax.set_xlabel('Real ($)'); ax.set_ylabel('Predicho ($)'); ax.set_title(name); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
fi = pd.Series(best_gb.feature_importances_, index=X.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 5))
fi.plot(kind='barh', ax=ax, color=COLORS[0], alpha=0.85)
ax.set_title('Importancia de Variables — Gradient Boosting')
plt.tight_layout(); plt.show()

In [ ]:
# Residuos
residuos = y_test.values - y_pred_gb
fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].scatter(y_pred_gb, residuos, alpha=0.35, color=COLORS[0], s=20)
axes[0].axhline(0, color=COLORS[3], linestyle='--')
axes[0].set_xlabel('Predichos ($)'); axes[0].set_ylabel('Residuos')
axes[0].set_title('Residuos vs Predichos')
axes[1].hist(residuos, bins=35, color=COLORS[1], edgecolor='white', alpha=0.85)
axes[1].axvline(0, color=COLORS[3], linestyle='--')
axes[1].set_title('Distribución de Residuos')
plt.tight_layout(); plt.show()

## 🏆 Conclusión

El modelo **Gradient Boosting** logró el mejor desempeño (R²≈0.80), superando a la regresión lineal y KNN.

Las variables más influyentes fueron: `nivel_membresia`, `items_en_carrito`, `paginas_vistas_sesion` y `frecuencia_visitas_mes`.

Se recomienda continuar el modelo con más datos reales y explorar XGBoost/LightGBM para mayor rendimiento.
